# 에이전트 루프의 놀라운 효과

이번 노트북에서는 **에이전트 루프(Agent Loop)** 의 핵심 구조를 이해하고, 단순한 루프가 어떻게 복잡한 문제를 해결할 수 있는지 실습합니다.

## 개요

| 주제 | 내용 |
|------|------|
| 에이전트 정의 | 3가지 관점에서 에이전트란 무엇인가 |
| 에이전트 루프 | 핵심 구조와 동작 원리 |
| Tool 시스템 | Todo 리스트 기반 도구 구현 |
| 실전 테스트 | 에이전트가 문제를 단계별로 풀어가는 과정 |

## 학습 목표

1. 에이전트의 다양한 정의를 비교하고 이해하기
2. 에이전트 루프의 핵심 구조 파악하기
3. Tool 함수와 JSON 스키마를 직접 만들어보기
4. 에이전트 루프를 처음부터 구현하기

---

## 핵심 아이디어

> "에이전트의 본질은 놀라울 정도로 단순합니다. **LLM이 도구를 루프 안에서 호출하며 목표를 달성하는 것** — 그것이 전부입니다."

---

## 1. 에이전트란 무엇인가?

"에이전트"라는 용어는 다양한 의미로 사용됩니다. 주요 정의를 비교해보겠습니다.

### 3가지 정의 비교

| 관점 | 정의 | 특징 |
|------|------|------|
| **학술적 정의** | 환경을 인식하고 자율적으로 행동하는 개체 | 가장 넓은 의미, AI 전반 포괄 |
| **프레임워크 정의** | LLM을 활용하여 워크플로우를 오케스트레이션하는 시스템 | LangChain, AutoGen 등이 사용 |
| **실용적 정의** | 루프 안에서 도구를 사용하여 목표를 달성하는 LLM | **이번 노트북의 초점** |

### 이번 노트북에서의 에이전트

> **"LLM 에이전트는 도구를 루프 안에서 실행하며 목표를 달성한다."**

이 정의가 강력한 이유:
- 구현이 단순합니다 (while 루프 + Tool 호출)
- 하지만 결과는 놀라울 정도로 강력합니다
- 대부분의 실제 에이전트 시스템이 이 패턴을 따릅니다

---

## 2. 에이전트 루프 아키텍처

에이전트 루프의 핵심 구조를 시각화하면 다음과 같습니다.

```
┌──────────────────────────────────────────────────────────────────┐
│                      에이전트 루프 구조                          │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│  ┌──────────────┐                                               │
│  │  사용자 요청  │                                               │
│  └──────┬───────┘                                               │
│         │                                                        │
│         ▼                                                        │
│  ┌─────────────────────────────────────────────────────┐        │
│  │                  while 루프                         │        │
│  │                                                     │        │
│  │  ┌───────────┐     ┌──────────────┐     ┌────────┐ │        │
│  │  │   LLM     │────▶│ finish_reason│────▶│ Tool   │ │        │
│  │  │  API 호출  │     │  확인        │     │ 실행   │ │        │
│  │  └───────────┘     └──────────────┘     └───┬────┘ │        │
│  │       ▲                                      │      │        │
│  │       └──── 결과를 messages에 추가 ───────────┘      │        │
│  │                                                     │        │
│  │  finish_reason == "stop" → 루프 종료               │        │
│  └─────────────────────────────────────────────────────┘        │
│         │                                                        │
│         ▼                                                        │
│  ┌──────────────┐                                               │
│  │  최종 응답    │                                               │
│  └──────────────┘                                               │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

### 왜 이 단순한 루프가 강력한가?

1. **LLM이 스스로 계획을 세웁니다** — 복잡한 문제를 작은 단계로 분해
2. **각 단계의 결과를 관찰합니다** — Tool 실행 결과를 보고 다음 행동 결정
3. **자동으로 반복합니다** — 목표를 달성할 때까지 계속 진행
4. **유연합니다** — 같은 루프 구조로 다양한 문제를 풀 수 있음

---

## 3. 환경 설정

In [2]:
# 환경 설정 및 라이브러리 임포트
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print("API key found.")
else:
    print("No API key was found")

MODEL = "gpt-4o-mini"
openai = OpenAI()

API key found.


In [3]:
# rich 라이브러리 — 터미널 출력을 보기 좋게 포맷팅
# pip install rich

from rich.console import Console
from rich.markdown import Markdown

console = Console()

def show(text):
    """마크다운 형식의 텍스트를 보기 좋게 출력합니다."""
    console.print(Markdown(text))

# 테스트
show("**rich** 라이브러리로 *마크다운*을 예쁘게 출력할 수 있습니다.")

rich 라이브러리로 마크다운을 예쁘게 출력할 수 있습니다.

---

## 4. Todo 리스트 시스템 구현

에이전트가 사용할 도구로 **Todo 리스트 시스템**을 만들겠습니다. 에이전트는 이 도구를 사용하여:

1. 해야 할 일 목록을 만들고
2. 하나씩 완료 처리하며
3. 현재 진행 상황을 확인합니다

### 함수 목록

| 함수명 | 역할 | Tool로 제공? |
|--------|------|-------------|
| `get_todo_report` | 현재 Todo 상태 보고서 반환 | 아니오 (내부 용도) |
| `create_todos` | 새로운 할 일 목록 추가 | 예 |
| `mark_complete` | 특정 항목 완료 처리 | 예 |

In [4]:
# Todo 리스트 상태 관리

todos = []       # 할 일 목록
completed = []   # 완료된 항목

def get_todo_report():
    """현재 Todo 상태 보고서를 문자열로 반환합니다."""
    report = "## 현재 Todo 상태\n\n"
    for i, todo in enumerate(todos, 1):
        status = "완료" if todo in completed else "미완료"
        marker = "~~" if todo in completed else ""
        report += f"{i}. {marker}{todo}{marker} [{status}]\n"
    if not todos:
        report += "(아직 할 일이 없습니다)\n"
    return report

def create_todos(descriptions):
    """새로운 할 일 항목들을 추가합니다."""
    for desc in descriptions:
        todos.append(desc)
    return {"status": "success", "message": f"{len(descriptions)}개의 할 일이 추가되었습니다.", "report": get_todo_report()}

def mark_complete(index, completion_notes=""):
    """특정 번호의 할 일을 완료 처리합니다. (1-based index)"""
    if 1 <= index <= len(todos):
        todo = todos[index - 1]
        completed.append(todo)
        result = f"'{todo}' 완료 처리됨."
        if completion_notes:
            result += f" 메모: {completion_notes}"
        return {"status": "success", "message": result, "report": get_todo_report()}
    else:
        return {"status": "error", "message": f"잘못된 인덱스: {index}. 범위: 1-{len(todos)}"}

In [5]:
# 함수 테스트 — 직접 호출하여 동작 확인

print("=== 초기 상태 ===")
print(get_todo_report())

print("=== Todo 추가 ===")
result = create_todos(["파이썬 공부하기", "보고서 작성하기", "운동하기"])
print(json.dumps(result, ensure_ascii=False, indent=2))

print("\n=== 2번 항목 완료 ===")
result = mark_complete(2, "드래프트 완성")
print(json.dumps(result, ensure_ascii=False, indent=2))

# 상태 초기화 (에이전트 테스트를 위해)
todos = []
completed = []

=== 초기 상태 ===
## 현재 Todo 상태

(아직 할 일이 없습니다)

=== Todo 추가 ===
{
  "status": "success",
  "message": "3개의 할 일이 추가되었습니다.",
  "report": "## 현재 Todo 상태\n\n1. 파이썬 공부하기 [미완료]\n2. 보고서 작성하기 [미완료]\n3. 운동하기 [미완료]\n"
}

=== 2번 항목 완료 ===
{
  "status": "success",
  "message": "'보고서 작성하기' 완료 처리됨. 메모: 드래프트 완성",
  "report": "## 현재 Todo 상태\n\n1. 파이썬 공부하기 [미완료]\n2. ~~보고서 작성하기~~ [완료]\n3. 운동하기 [미완료]\n"
}


---

## 5. Tool JSON 스키마 정의

LLM이 Tool을 사용하려면 **JSON 스키마**로 함수 구조를 알려줘야 합니다.

### 스키마 작성 가이드

| 필드 | 역할 | 팁 |
|------|------|----|
| `name` | 함수명 | Python 함수명과 정확히 일치 |
| `description` | 함수 설명 | 상세할수록 LLM이 적절히 호출 |
| `parameters` | 파라미터 정의 | 타입, 설명, 필수 여부 명시 |
| `required` | 필수 파라미터 | 반드시 전달해야 하는 값 |

In [6]:
# Tool JSON 스키마 정의

create_todos_json = {
    "type": "function",
    "function": {
        "name": "create_todos",
        "description": "할 일 목록에 새로운 항목들을 추가합니다. 계획을 세우거나 작업을 단계별로 분해할 때 사용하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "descriptions": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "추가할 할 일 항목들의 설명 목록"
                }
            },
            "required": ["descriptions"],
            "additionalProperties": False
        }
    }
}

mark_complete_json = {
    "type": "function",
    "function": {
        "name": "mark_complete",
        "description": "특정 번호의 할 일을 완료 처리합니다. 작업을 수행한 후 그 결과를 메모와 함께 기록하세요.",
        "parameters": {
            "type": "object",
            "properties": {
                "index": {
                    "type": "integer",
                    "description": "완료할 항목의 번호 (1부터 시작)"
                },
                "completion_notes": {
                    "type": "string",
                    "description": "완료 시 작성한 메모 — 수행 결과나 핵심 내용"
                }
            },
            "required": ["index", "completion_notes"],
            "additionalProperties": False
        }
    }
}

tools = [create_todos_json, mark_complete_json]

# 스키마 확인
print(json.dumps(tools, indent=2, ensure_ascii=False))

[
  {
    "type": "function",
    "function": {
      "name": "create_todos",
      "description": "할 일 목록에 새로운 항목들을 추가합니다. 계획을 세우거나 작업을 단계별로 분해할 때 사용하세요.",
      "parameters": {
        "type": "object",
        "properties": {
          "descriptions": {
            "type": "array",
            "items": {
              "type": "string"
            },
            "description": "추가할 할 일 항목들의 설명 목록"
          }
        },
        "required": [
          "descriptions"
        ],
        "additionalProperties": false
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "mark_complete",
      "description": "특정 번호의 할 일을 완료 처리합니다. 작업을 수행한 후 그 결과를 메모와 함께 기록하세요.",
      "parameters": {
        "type": "object",
        "properties": {
          "index": {
            "type": "integer",
            "description": "완료할 항목의 번호 (1부터 시작)"
          },
          "completion_notes": {
            "type": "string",
            "description": "완료 시 작성한 메모 — 수행 결과나 핵심 내용"
  

---

## 6. Tool 호출 핸들러

In [7]:
# Tool 호출 핸들러 — globals() 기반 동적 디스패치

def handle_tool_calls(tool_calls):
    """LLM의 Tool 호출 요청을 처리하고 결과를 반환합니다."""
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"  [Tool 호출] {tool_name}({json.dumps(arguments, ensure_ascii=False)})")
        
        # globals()에서 함수를 동적으로 찾아 실행
        tool = globals().get(tool_name)
        if tool:
            result = tool(**arguments)
        else:
            result = {"error": f"알 수 없는 Tool: {tool_name}"}
        
        results.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result, ensure_ascii=False)
        })
    return results

---

## 7. 에이전트 루프 구현

이제 핵심인 **에이전트 루프**를 구현합니다. `loop()` 함수의 동작을 단계별로 살펴보겠습니다.

### 동작 순서

1. `messages`를 LLM API에 전달하여 응답 받기
2. `finish_reason` 확인:
   - `"tool_calls"` → Tool 실행 → 결과를 messages에 추가 → 1번으로 돌아감
   - `"stop"` → 최종 응답 출력 → 루프 종료
3. 각 단계에서 현재 Todo 상태를 출력하여 진행 과정 관찰

### 핵심 포인트

- LLM이 **자체적으로 계획을 수립**합니다 (create_todos 호출)
- 각 단계를 **순서대로 실행**합니다 (mark_complete 호출)
- **목표 달성 시 자동 종료**합니다 (모든 Todo 완료 시)

In [9]:
# 에이전트 루프 함수

def loop(messages, max_iterations=20):
    """에이전트 루프 — Tool을 반복 호출하며 목표를 달성합니다."""
    iteration = 0
    done = False
    
    while not done and iteration < max_iterations:
        iteration += 1
        print(f"\n{'='*60}")
        print(f"[반복 {iteration}]")
        
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        choice = response.choices[0]
        
        if choice.finish_reason == "tool_calls":
            # assistant 메시지 추가
            messages.append(choice.message)
            
            # Tool 실행 및 결과 추가
            tool_results = handle_tool_calls(choice.message.tool_calls)
            messages.extend(tool_results)
            
            # 현재 Todo 상태 출력
            show(get_todo_report())
        else:
            # 최종 응답
            print("\n[에이전트 완료]")
            show(choice.message.content)
            done = True
    
    if not done:
        print(f"\n최대 반복 횟수({max_iterations})에 도달했습니다.")
    
    return messages

---

## 8. 실전 테스트: 기차 문제

에이전트에게 수학 문제를 풀도록 해보겠습니다. 에이전트는:

1. 문제를 분석하고 풀이 계획을 세웁니다 (create_todos)
2. 각 단계를 실행하며 결과를 기록합니다 (mark_complete)
3. 모든 단계가 완료되면 최종 답변을 제공합니다

### 문제

> 서울에서 부산으로 가는 기차가 오후 2시에 출발하여 시속 120km로 달립니다. 부산에서 서울로 오는 기차는 오후 3시에 출발하여 시속 80km로 달립니다. 서울-부산 거리가 400km일 때, 두 기차가 만나는 시각은 언제인가요?

In [10]:
# 상태 초기화
todos = []
completed = []

# 시스템 프롬프트
system_message = """당신은 문제 해결 에이전트입니다.

주어진 문제를 풀기 위해 다음 절차를 따르세요:
1. 먼저 create_todos를 사용하여 풀이 계획을 단계별로 수립하세요.
2. 각 단계를 직접 수행한 후, mark_complete로 완료 처리하고 결과를 메모하세요.
3. 모든 단계를 완료한 후 최종 답변을 제공하세요.

반드시 단계별로 진행하세요. 한 번에 모든 것을 처리하지 마세요."""

# 사용자 문제
user_message = """서울에서 부산으로 가는 기차가 오후 2시에 출발하여 시속 120km로 달립니다.
부산에서 서울로 오는 기차는 오후 3시에 출발하여 시속 80km로 달립니다.
서울-부산 거리가 400km일 때, 두 기차가 만나는 시각은 언제인가요?"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

# 에이전트 루프 실행
result = loop(messages)


[반복 1]
  [Tool 호출] create_todos({"descriptions": ["계산에 필요한 변수를 정의하기", "서울에서 부산으로 가는 기차의 이동 거리 식을 작성하기", "부산에서 서울로 가는 기차의 이동 거리 식을 작성하기", "두 기차의 만나는 시각을 계산하기"]})


현재 Todo 상태                                                   

 1 계산에 필요한 변수를 정의하기 [미완료]                                                                          
 2 서울에서 부산으로 가는 기차의 이동 거리 식을 작성하기 [미완료]                                                  
 3 부산에서 서울로 가는 기차의 이동 거리 식을 작성하기 [미완료]                                                    
 4 두 기차의 만나는 시각을 계산하기 [미완료]


[반복 2]
  [Tool 호출] mark_complete({"index": 1, "completion_notes": "문제를 해결하기 위한 변수를 정의했다: 서울-부산 거리 = 400km, 기차1 속도 = 120km/h, 기차2 속도 = 80km/h, 기차1 출발시간 = 14시, 기차2 출발시간 = 15시."})


현재 Todo 상태                                                   

 1 계산에 필요한 변수를 정의하기 [완료]                                                                            
 2 서울에서 부산으로 가는 기차의 이동 거리 식을 작성하기 [미완료]                                                  
 3 부산에서 서울로 가는 기차의 이동 거리 식을 작성하기 [미완료]                                                    
 4 두 기차의 만나는 시각을 계산하기 [미완료]


[반복 3]
  [Tool 호출] mark_complete({"index": 2, "completion_notes": "서울에서 부산으로 가는 기차의 이동 거리 식을 작성했다: 거리 = 속도 × 시간, 즉 D1 = 120t (t는 기차1의 이동 시간)."})


...

현재 Todo 상태                                                   

 1 계산에 필요한 변수를 정의하기 [완료]                                                                            
 2 서울에서 부산으로 가는 기차의 이동 거리 식을 작성하기 [완료]                                                    
 3 부산에서 서울로 가는 기차의 이동 거리 식을 작성하기 [완료]                                                      
 4 두 기차의 만나는 시각을 계산하기 [미완료]


[반복 5]
  [Tool 호출] mark_complete({"index": 4, "completion_notes": "두 기차의 만나는 시각을 계산했다: 120t + 80(t-1) = 400을 풀어 t를 구하고, 이를 통해 만나는 시각을 산출했다. t = 2.5시간, 따라서 기차가 만나는 시각은 오후 4시 30분이다."})


현재 Todo 상태                                                   

 1 계산에 필요한 변수를 정의하기 [완료]                                                                            
 2 서울에서 부산으로 가는 기차의 이동 거리 식을 작성하기 [완료]                                                    
 3 부산에서 서울로 가는 기차의 이동 거리 식을 작성하기 [완료]                                                      
 4 두 기차의 만나는 시각을 계산하기 [완료]


[반복 6]

[에이전트 완료]


두 기차가 만나는 시각은 오후 4시 30분입니다.                                                                       

계산 과정은 다음과 같습니다:                                                                                       

 1 서울에서 부산으로 가는 기차의 이동 거리: ( D_1 = 120t )                                                         
 2 부산에서 서울로 가는 기차의 이동 거리: ( D_2 = 80(t - 1) )                                                      
 3 두 기차의 거리 합이 400km가 되어야 하므로, 식을 세웁니다: [ 120t + 80(t - 1) = 400 ]                            
 4 이 식을 풀면 ( t = 2.5 )시간이 나오고, 이는 오후 2시에 출발한 기차의 출발시간에 더하면 오후 4시 30분이 됩니다.

---

## 9. 추가 예제: 요리 레시피 계획

같은 에이전트 루프로 완전히 다른 유형의 문제도 풀 수 있습니다. 이번에는 요리 레시피를 계획하는 에이전트를 만들어보겠습니다.

In [11]:
# 상태 초기화
todos = []
completed = []

# 요리 에이전트
system_message_cook = """당신은 요리 계획 에이전트입니다.

사용자가 요리를 요청하면:
1. create_todos로 재료 준비, 조리 과정 등을 단계별 계획으로 수립하세요.
2. 각 단계를 설명한 후 mark_complete로 완료 처리하세요. completion_notes에 상세한 조리 방법을 적어주세요.
3. 모든 단계가 끝나면 완성된 레시피를 요약해주세요.

반드시 한 단계씩 진행하세요."""

user_message_cook = "김치찌개를 만드는 방법을 단계별로 알려주세요. 2인분 기준으로요."

messages_cook = [
    {"role": "system", "content": system_message_cook},
    {"role": "user", "content": user_message_cook}
]

# 에이전트 루프 실행
result_cook = loop(messages_cook)


[반복 1]
  [Tool 호출] create_todos({"descriptions": ["재료 준비: 돼지고기(목살 또는 삼겹살) 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루 1큰술, 간장 1큰술, 소금, 후추", "재료를 잘라 준비: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자르기", "냄비에 돼지고기를 넣고 볶기: 중불에서 돼지고기를 볶아 겉면이 익을 때까지 조리하기", "김치를 추가하고 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶기", "물 넣기: 볶은 재료에 물(500ml 정도)을 넣고 끓이기 시작하기", "양념 추가하기: 고추가루, 간장을 넣고 잘 섞은 후 끓이기", "두부 및 대파 추가: 두부와 대파를 넣고, 나머지 재료와 함께 10분간 끓이기", "마늘 넣기: 마지막으로 다진 마늘을 넣고 간을 보고, 소금과 후추로 맛 조절하기", "완성된 김치찌개를 그릇에 담기"]})


현재 Todo 상태                                                   

  1 재료 준비: 돼지고기(목살 또는 삼겹살) 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루      
    1큰술, 간장 1큰술, 소금, 후추 [미완료]                                                                         
  2 재료를 잘라 준비: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자르기 [미완료]         
  3 냄비에 돼지고기를 넣고 볶기: 중불에서 돼지고기를 볶아 겉면이 익을 때까지 조리하기 [미완료]                     
  4 김치를 추가하고 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶기 [미완료]                                        
  5 물 넣기: 볶은 재료에 물(500ml 정도)을 넣고 끓이기 시작하기 [미완료]                                            
  6 양념 추가하기: 고추가루, 간장을 넣고 잘 섞은 후 끓이기 [미완료]                                                
  7 두부 및 대파 추가: 두부와 대파를 넣고, 나머지 재료와 함께 10분간 끓이기 [미완료]                               
  8 마늘 넣기: 마지막으로 다진 마늘을 넣고 간을 보고, 소금과 후추로 맛 조절하기 [미완료]                           
  9 완성된 김치찌개를 그릇에 담기 [미완료]


[반복 2]
  [Tool 호출] mark_complete({"index": 1, "completion_notes": "재료를 준비했습니다: 돼지고기 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루 1큰술, 간장 1큰술, 소금, 후추."})


현재 Todo 상태                                                   

  1 재료 준비: 돼지고기(목살 또는 삼겹살) 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루      
    1큰술, 간장 1큰술, 소금, 후추 [완료]                                                                           
  2 재료를 잘라 준비: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자르기 [미완료]         
  3 냄비에 돼지고기를 넣고 볶기: 중불에서 돼지고기를 볶아 겉면이 익을 때까지 조리하기 [미완료]                     
  4 김치를 추가하고 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶기 [미완료]                                        
  5 물 넣기: 볶은 재료에 물(500ml 정도)을 넣고 끓이기 시작하기 [미완료]                                            
  6 양념 추가하기: 고추가루, 간장을 넣고 잘 섞은 후 끓이기 [미완료]                                                
  7 두부 및 대파 추가: 두부와 대파를 넣고, 나머지 재료와 함께 10분간 끓이기 [미완료]                               
  8 마늘 넣기: 마지막으로 다진 마늘을 넣고 간을 보고, 소금과 후추로 맛 조절하기 [미완료]                           
  9 완성된 김치찌개를 그릇에 담기 [미완료]


[반복 3]
  [Tool 호출] mark_complete({"index": 2, "completion_notes": "모든 재료를 잘라 준비했습니다: 돼지고기, 양파, 대파를 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 잘랐습니다."})


...

현재 Todo 상태                                                   

  1 재료 준비: 돼지고기(목살 또는 삼겹살) 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루      
    1큰술, 간장 1큰술, 소금, 후추 [완료]                                                                           
  2 재료를 잘라 준비: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자르기 [완료]           
  3 냄비에 돼지고기를 넣고 볶기: 중불에서 돼지고기를 볶아 겉면이 익을 때까지 조리하기 [완료]                       
  4 김치를 추가하고 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶기 [완료]                                          
  5 물 넣기: 볶은 재료에 물(500ml 정도)을 넣고 끓이기 시작하기 [완료]                                              
  6 양념 추가하기: 고추가루, 간장을 넣고 잘 섞은 후 끓이기 [완료]                                                  
  7 두부 및 대파 추가: 두부와 대파를 넣고, 나머지 재료와 함께 10분간 끓이기 [완료]                                 
  8 마늘 넣기: 마지막으로 다진 마늘을 넣고 간을 보고, 소금과 후추로 맛 조절하기 [완료]                             
  9 완성된 김치찌개를 그릇에 담기 [미완료]


[반복 10]
  [Tool 호출] mark_complete({"index": 9, "completion_notes": "완성된 김치찌개를 그릇에 담았습니다."})


현재 Todo 상태                                                   

  1 재료 준비: 돼지고기(목살 또는 삼겹살) 200g, 김치 200g, 두부 1/2모, 양파 1개, 대파 1대, 마늘 3쪽, 고추가루      
    1큰술, 간장 1큰술, 소금, 후추 [완료]                                                                           
  2 재료를 잘라 준비: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자르기 [완료]           
  3 냄비에 돼지고기를 넣고 볶기: 중불에서 돼지고기를 볶아 겉면이 익을 때까지 조리하기 [완료]                       
  4 김치를 추가하고 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶기 [완료]                                          
  5 물 넣기: 볶은 재료에 물(500ml 정도)을 넣고 끓이기 시작하기 [완료]                                              
  6 양념 추가하기: 고추가루, 간장을 넣고 잘 섞은 후 끓이기 [완료]                                                  
  7 두부 및 대파 추가: 두부와 대파를 넣고, 나머지 재료와 함께 10분간 끓이기 [완료]                                 
  8 마늘 넣기: 마지막으로 다진 마늘을 넣고 간을 보고, 소금과 후추로 맛 조절하기 [완료]                             
  9 완성된 김치찌개를 그릇에 담기 [완료]


[반복 11]

[에이전트 완료]


김치찌개 레시피가 완성되었습니다! 2인분 기준의 조리 과정을 요약하겠습니다.                                         

                                                   김치찌개 재료                                                   

 • 돼지고기 (목살 또는 삼겹살) 200g                                                                                
 • 김치 200g                                                                                                       
 • 두부 1/2모                                                                                                      
 • 양파 1개                                                                                                        
 • 대파 1대                                                                                                        
 • 마늘 3쪽                                                                                                        
 • 고추가루 1큰술                                                                                                  
 • 간장 1큰술                                                                                                      
 • 소금, 후추 (약간)                                                                                               

                                                     조리 과정                                                     

  1 재료 준비: 모든 재료를 준비합니다.                                                                             
  2 재료 자르기: 돼지고기, 양파, 대파는 먹기 좋은 크기로 자르고, 두부는 큐브 모양으로 자릅니다.                    
  3 돼지고기 볶기: 냄비에 돼지고기를 넣고 중불에서 볶아 겉면이 익을 때까지 조리합니다.                             
  4 김치 추가 볶기: 볶은 돼지고기에 김치를 넣고 3~4분간 볶습니다.                                                  
  5 물 넣기: 볶은 재료에 물(약 500ml)을 넣고 끓이기 시작합니다.                                                    
  6 양념 추가하기: 고추가루와 간장을 넣고 잘 섞은 후 끓입니다.                                                     
  7 두부 및 대파 추가: 두부와 대파를 넣고 함께 10분간 끓입니다.                                                    
  8 마늘 넣기 및 간 보기: 마지막으로 다진 마늘을 넣고 간을 보며 소금과 후추로 맛을 조절합니다.                     
  9 완성된 김치찌개 담기: 김치찌개를 그릇에 담아 제공합니다.                                                       

맛있게 드세요! 🍲

---

## 10. 핵심 정리

### 에이전트 루프 패턴 요약

```
┌─────────────────────────────────────────────────────────────┐
│                  에이전트 루프 핵심 요소                      │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. Tool 정의     Python 함수 + JSON 스키마                 │
│                                                             │
│  2. 핸들러        globals() 기반 동적 디스패치               │
│                                                             │
│  3. 루프          while + finish_reason 확인                │
│                                                             │
│  4. 상태 관리     messages 리스트에 대화 이력 누적           │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### 실무 적용 시 고려사항

| 항목 | 고려사항 |
|------|----------|
| **최대 반복 횟수** | 무한 루프 방지를 위해 반드시 설정 |
| **비용 관리** | 각 반복마다 API 호출 비용 발생 |
| **에러 처리** | Tool 실행 실패 시 graceful한 처리 필요 |
| **로깅** | 디버깅을 위해 각 단계의 입출력 기록 |
| **모델 선택** | 복잡한 추론에는 더 강력한 모델 고려 |
| **컨텍스트 윈도우** | 반복이 많아지면 토큰 한도 주의 |

### 핵심 메시지

> "에이전트의 힘은 복잡한 프레임워크가 아니라, **LLM + Tool + 루프**라는 단순한 조합에서 나옵니다. 이 패턴을 이해하면 어떤 에이전트 프레임워크든 쉽게 이해할 수 있습니다."

---

## 연습 과제

### 과제 1: 에이전트 루프 처음부터 만들기

이 노트북의 코드를 참고하지 않고, 에이전트 루프를 **처음부터** 구현해보세요.

1. 자신만의 Tool 함수 2-3개 정의하기
2. JSON 스키마 작성하기
3. handle_tool_calls 핸들러 구현하기
4. 에이전트 루프 함수 구현하기
5. 테스트 실행하기

### 과제 2: 다양한 시나리오 테스트

같은 에이전트 루프에 다른 시스템 프롬프트와 사용자 메시지를 주어 다양한 문제를 풀어보세요.

- 여행 계획 세우기
- 프로젝트 관리
- 학습 계획 수립

### 과제 3: Tool 확장

현재 `create_todos`와 `mark_complete` 외에 새로운 Tool을 추가해보세요.

- `delete_todo` — 특정 항목 삭제
- `reorder_todos` — 우선순위 변경
- `web_search` — 실제 웹 검색 연동

---

## 참고 자료

- [OpenAI Function Calling 문서](https://platform.openai.com/docs/guides/function-calling)
- [Building Effective Agents - Anthropic](https://www.anthropic.com/research/building-effective-agents)
- [Rich 라이브러리 문서](https://rich.readthedocs.io/)